<a href="https://colab.research.google.com/github/prithwis/Centaur/blob/main/MOS_Simulation_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/images/CentaurBanner2.png)<br>


<hr>

[Prithwis Mukerjee](http://www.linkedin.com/in/prithwis)<br>

## Middle East Oil Security Simulation Engine
MOS is an LLM-driven geopolitical simulation in which strategic actors choose actions within an explicitly represented and evolving WorldState. Chanakya proposes actor-specific strategic options, Centaur evaluates their relative likelihood, and ZeitWorld translates the selected Event into the next WorldState. The objective is not to predict the future, but to explore plausible, path-dependent trajectories arising from alternative strategic choices.

In [1]:
!wget -O mos.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mos.py
!wget -O mos2.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mos2.py
!wget -O mos3.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mos3.py

from mos import *
from mos2 import *
from mos3 import *

In [ ]:
import importlib

import mos
import mos2
import mos3

importlib.reload(mos)
importlib.reload(mos2)
importlib.reload(mos3)

from mos import *
from mos2 import *
from mos3 import *

In [2]:
client = authenticateOpenAI()
model = "gpt-5.6-luna"

OpenAI authentication successful ✔
Logged in as Calcutta prithwis@yantrajaal.com


## WorldState and ZeitWorld

In [3]:
# These are the eight variables that currently define the World
# This number can be increased if necessasary
#
# ---------------------------------------------------------------

worldVariables = [

    WorldVariable(
        name="Oil Price",
        description=(
            "International crude oil price level, represented "
            "by the Brent benchmark."
        ),
        direction="Higher index means higher oil price."
    ),

    WorldVariable(
        name="Hormuz Oil Flow",
        description=(
            "Volume of crude oil and petroleum products successfully "
            "transiting the Strait of Hormuz."
        ),
        direction="Higher index means greater oil flow."
    ),

    WorldVariable(
        name="India Gulf Oil Imports",
        description=(
            "Volume of crude oil imported by India from "
            "Gulf-origin oil-producing countries."
        ),
        direction=(
            "Higher index means greater Indian imports "
            "of Gulf-origin oil."
        )
    ),

    WorldVariable(
        name="Iran Sanctions Enforcement",
        description=(
            "Degree to which restrictions on Iranian petroleum exports, "
            "trade and financial transactions are effectively enforced "
            "rather than circumvented."
        ),
        direction=(
            "Higher index means more effective sanctions enforcement "
            "and less successful circumvention."
        )
    ),

    WorldVariable(
        name="US/Allied Cohesion",
        description=(
            "Degree of alignment between the United States and cooperating "
            "states in their response to Iran and the Gulf crisis."
        ),
        direction="Higher index means greater US/allied cohesion."
    ),

    WorldVariable(
        name="Iran Internal Cohesion",
        description=(
            "Degree of alignment within Iran's political, military and "
            "security establishment in responding to the crisis."
        ),
        direction="Higher index means greater internal Iranian cohesion."
    ),

    WorldVariable(
        name="US-Iran Tension",
        description=(
            "Intensity of strategic hostility between the United States "
            "and Iran across diplomatic, economic and military domains."
        ),
        direction="Higher index means greater US-Iran tension."
    ),

    WorldVariable(
        name="Regional Armed Conflict",
        description=(
            "Intensity and geographic breadth of kinetic military activity "
            "associated with the regional crisis."
        ),
        direction=(
            "Higher index means more intense or geographically "
            "widespread armed conflict."
        )
    )
]

In [4]:
# Historical Data and
# LLM Roles and prompts are stored and retrieved from Google Drive
#
# ----------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

MOS_HOME = "/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/MOS/"
WORLDFACTS_DIR = f"{MOS_HOME}/WorldFacts"
ZW_PATH = f"{MOS_HOME}/ZeitWorld"
CH_PATH = f"{MOS_HOME}/Chanakya"

!ls -l "{WORLDFACTS_DIR}"
!ls -lh "{WORLDFACTS_DIR}/MD"
!ls -lh "{WORLDFACTS_DIR}/NEWS"
!ls -lh "{ZW_PATH}"
!ls -lh "{CH_PATH}"

Mounted at /content/drive
total 12
drwx------ 2 root root 4096 Sep 28 09:09 MD
drwx------ 2 root root 4096 Sep 29 09:42 NEWS
drwx------ 2 root root 4096 Sep 28 06:55 PDF
total 159K
-rw------- 1 root root 7.7K Oct  1 09:31 IranData_261001.md
-rw------- 1 root root 6.9K Oct  1 09:31 MissilesofIranMissil_261001.md
-rw------- 1 root root 8.1K Oct  1 09:32 OilMarketReportSepte_261001.md
-rw------- 1 root root  13K Oct  1 09:32 PressReleasePagePres_261001.md
-rw------- 1 root root 4.3K Oct  1 09:33 ShortTermEnergyOutlo_261001.md
-rw------- 1 root root  31K Oct  1 09:34 SnapshotofIndiasOilG_261001.md
-rw------- 1 root root  18K Oct  1 09:35 UKCommonsIranin2026S_261001.md
-rw------- 1 root root  70K Oct  1 09:37 USCongressR488873_261001.md
total 56K
-rw------- 1 root root 56K Oct  1 09:39 MOS-MC-LiveFeed_261001_1509_IST.txt
total 13K
-rw------- 1 root root 2.2K Oct  1 00:10 ZW_Initialisation_Prompt_v0.txt
-rw------- 1 root root 3.1K Oct  1 14:08 ZW_Initialisation_Prompt_v1.txt
-rw------- 1 roo

In [5]:
# Knowledge of the initial world state is collected from a
# a set of PDF files that have been converted to markdown format
# live news data from 1 source, viz. MoneyControl.com
# Assembled into one large corpus
#
# ------------------------------------------------

WorldFactBase = getWorldFactBase(WORLDFACTS_DIR)


Static sources : 8
Live sources   : 1
Characters     : 215,849
Words          : 32,422


In [6]:
# Zeitworld Roles and Prompts are retrieved from Google Drive

ZW_Role = getPrompt(f"{ZW_PATH}/ZW_Role_v0.txt")

#ZW_Initialisation_Prompt = getPrompt(f"{ZW_PATH}/ZW_Initialisation_Prompt_v0.txt")
ZW_Initialisation_Prompt = getPrompt(f"{ZW_PATH}/ZW_Initialisation_Prompt_v1.txt")

ZW_Update_Prompt = getPrompt(f"{ZW_PATH}/ZW_Update_Prompt_v0.txt")

In [7]:
# ZeitWorld obect is instantiated from its class along with its ROLE
#
ZeitWorld = ZeitWorld_Agent(client, model, ZW_Role)

#----------
print(ZeitWorld.initialised)
#print(ZeitWorld.model)

False


In [8]:
# W is a list of evolving World States
# An empty list is created. A blank world template is created, without data
# ----------------------------

W = []

W0_Template = WorldState(
    state_id="W0",
    variables=worldVariables
)

In [9]:
# ZeitWorld reads WorldFactBase data corpus and creates a World State based on World Template
# This initial World State is inserted into the World List W and becomes W[0]
#
# ----------------------------------------

#W[0] = ZeitWorld.initialise(W[0], WorldFactBase, ZW_Initialisation_Prompt)
_ = ZeitWorld.initialise(W,W0_Template, WorldFactBase, ZW_Initialisation_Prompt)

print(ZeitWorld.initialised)

Initialising WorldState ... 
Model: gpt-5.6-luna | Effort: medium | Input: 52,111 | Output: 2,071 | Reasoning: 152 | Total: 54,182
True


In [10]:
# A first look at the intial World State W[0]
# The initial values are all set to 100, as the baseline
#

#W[0].show(-2)  # ---- just the world vectors
#W[0].show()    # ---- the entire state
#W[0].show(4)    # ---- details of any one vector

## Turn and Events with Chanakya

In [11]:
# E is a list of events that will define History
#
E = []

In [12]:
# ============================================================
# Load Chanakya Roles
# ============================================================

#-rw------- 1 root root 2.0K Oct  2 00:06 CH_Role_Generic_v0.txt
#-rw------- 1 root root 1.8K Oct  2 00:07 CH_Role_Iran_v0.txt
#-rw------- 1 root root 1.8K Oct  2 00:07 CH_Role_US_v0.txt

CH_Generic_Role = getPrompt(f"{CH_PATH}/CH_Role_Generic_v0.txt")
CH_US_Role      = getPrompt(f"{CH_PATH}/CH_Role_US_v0.txt")
CH_IR_Role      = getPrompt(f"{CH_PATH}/CH_Role_Iran_v0.txt")
CH_BinaryHD     = getPrompt(f"{CH_PATH}/CH_Strategy_BinaryHD_v0.txt")

In [13]:
Chanakya_US = Chanakya_Agent(
    agent_id="ChanakyaUS_HD",
    actor="US",
    generic_role=CH_Generic_Role,
    specific_role=CH_US_Role,
    strategy=CH_BinaryHD
)

Chanakya_US.show()

Chanakya_IR = Chanakya_Agent(
    agent_id="ChanakyaIR_HD",
    actor="Iran",
    generic_role=CH_Generic_Role,
    specific_role=CH_IR_Role,
    strategy=CH_BinaryHD
)

Chanakya_IR.show()

ChanakyaRegistry = {
    "US": Chanakya_US,
    "Iran": Chanakya_IR
}

Chanakya : ChanakyaUS_HD
Actor    : US
Chanakya : ChanakyaIR_HD
Actor    : Iran


In [15]:
# Two optional Triggers, we manually choose any ONE

Trigger1 = Event(
    actor="Iran",
    action=(
        "Iran declares the Strait of Hormuz closed to all commercial shipping, "
        "orders its forces to prevent tanker transit, and launches attacks against "
        "US naval forces operating in the Gulf."
    ),
    rationale = "Initial Trigger for Iran"
)


Trigger2 = Event(
    actor="US",
    action=(
        "The United States announces an immediate suspension of offensive military "
        "operations against Iran and offers direct negotiations on maritime security "
        "and sanctions relief, while maintaining defensive protection of US forces "
        "and commercial shipping in the Gulf."
    ),
    rationale = "Initial Trigger for Iran"
)


In [16]:
DefaultTrigger = Trigger1
#Trigger = Trigger2

In [ ]:
# ============================================================
# Establish Current Trigger
# ============================================================

epoch = len(W) - 1
if len(E) == 0:
    CurrentTrigger = DefaultTrigger
    CurrentTrigger.event_id = f"E{epoch}"
    E.append(CurrentTrigger)
else:
    CurrentTrigger = E[-1]

E[epoch].show()

In [28]:
# Working with Current Trigger -- either manual, or from previous reaction
#
'''
epoch = len(W) - 1
CurrentTrigger.event_id = f"E{epoch}"
E.append(CurrentTrigger)
E[epoch].show()
'''

Event     : E1
Actor     : US
Action    : Expand military enforcement of the maritime exclusion effort by conducting targeted strikes against Iranian forces and infrastructure directly involved in attacks on US naval units and efforts to block tanker transit, while increasing coalition escort and patrol operations.
Rationale : Iran has directly attacked US forces and attempted to close a vital international waterway. A forceful response could impose immediate costs, restore US deterrence, protect military assets and shipping, and demonstrate to allies and Iran that coercive interference with Hormuz will not be accepted.


In [29]:
# ZeitWorld updates the world [W], moving it from W[epoch] to W[epoch+1] using Trigger Event E[epoch]
#
# -------------------------------

_ = ZeitWorld.update( W,E[epoch],ZW_Update_Prompt)

E[epoch].show()
W[epoch+1].show()

Updating WorldState ---------- W1
Model: gpt-5.6-luna | Effort: medium | Input: 3,521 | Output: 2,503 | Reasoning: 239 | Total: 6,024
Appended  WorldState ---------- W2
Event     : E1
Actor     : US
Action    : Expand military enforcement of the maritime exclusion effort by conducting targeted strikes against Iranian forces and infrastructure directly involved in attacks on US naval units and efforts to block tanker transit, while increasing coalition escort and patrol operations.
Rationale : Iran has directly attacked US forces and attempted to close a vital international waterway. A forceful response could impose immediate costs, restore US deterrence, protect military assets and shipping, and demonstrate to allies and Iran that coercive interference with Hormuz will not be accepted.

World State : W2
-------------------------------------------------------
 0  Oil Price                          118.00
 1  Hormuz Oil Flow                     57.00
 2  India Gulf Oil Imports           

In [ ]:
'''
def zgetActiveChanakya():

    last_actor = E[-1].actor

    if last_actor == "Iran":
        return Chanakya_US

    elif last_actor == "US":
        return Chanakya_IR

    else:
        raise ValueError(
            f"No responding Chanakya defined for actor: {last_actor}"
        )
'''

In [19]:
'''
def getActiveChanakya(last_event, chanakya_registry):

    if last_event.actor == "Iran":
        responding_actor = "US"

    elif last_event.actor == "US":
        responding_actor = "Iran"

    else:
        raise ValueError(
            f"No response rule defined for actor: {last_event.actor}"
        )

    return chanakya_registry[responding_actor]
'''

In [30]:
# ============================================================
# Turn Controller
# ============================================================

#ActiveChanakya = Chanakya_US
#ActiveChanakya = getActiveChanakya()
ActiveChanakya = getActiveChanakya(E[-1], ChanakyaRegistry)

ActiveChanakya.show()

Chanakya : ChanakyaIR_HD
Actor    : Iran


In [31]:
print(epoch)
E[epoch].show()
W[epoch+1].show()

1
Event     : E1
Actor     : US
Action    : Expand military enforcement of the maritime exclusion effort by conducting targeted strikes against Iranian forces and infrastructure directly involved in attacks on US naval units and efforts to block tanker transit, while increasing coalition escort and patrol operations.
Rationale : Iran has directly attacked US forces and attempted to close a vital international waterway. A forceful response could impose immediate costs, restore US deterrence, protect military assets and shipping, and demonstrate to allies and Iran that coercive interference with Hormuz will not be accepted.

World State : W2
-------------------------------------------------------
 0  Oil Price                          118.00
 1  Hormuz Oil Flow                     57.00
 2  India Gulf Oil Imports              91.00
 3  Iran Sanctions Enforcement         100.00
 4  US/Allied Cohesion                 108.00
 5  Iran Internal Cohesion             100.00
 6  US-Iran Tension 

In [32]:
# ============================================================
# Generate Candidate Reaction Events
# ============================================================

Candidates = ActiveChanakya.propose(
    client=client,
    model=model,
    trigger=E[epoch],
    world=W[epoch+1]
)

Chanakya working ---------- ChanakyaIR_HD
Model: gpt-5.6-luna | Effort: medium | Input: 2,483 | Output: 320 | Reasoning: 75 | Total: 2,803


In [33]:
for i, candidate in enumerate(Candidates):
    print(f"\nCandidate Reaction {i+1}")
    print("-" * 60)
    candidate.show()


Candidate Reaction 1
------------------------------------------------------------
Event     : 
Actor     : Iran
Action    : Maintain the declared closure of the Strait of Hormuz while intensifying calibrated asymmetric attacks on coalition naval, logistical, and regional support assets, using Iranian forces and aligned armed groups to impose costs without initiating unrestricted attacks on civilian shipping.
Rationale : Iran could seek to demonstrate that US strikes and expanded escorts cannot restore normal maritime access at acceptable cost. Sustained but calibrated pressure would preserve coercive leverage, reinforce deterrence, and improve Iran’s bargaining position while retaining some control over the level of escalation.

Candidate Reaction 2
------------------------------------------------------------
Event     : 
Actor     : Iran
Action    : Offer, through regional or other intermediaries, a reciprocal maritime de-escalation arrangement: Iran would suspend attacks on naval fo

In [ ]:
#print(Chanakya_US._event_to_text(E[t]))
#print()
#print(Chanakya_US._world_to_text(W[t+1]))

In [35]:
Reaction = Candidates[0]     # manual selection for now
Reaction.event_id = f"E{len(E)}"
E.append(Reaction)

In [36]:
print(CurrentTrigger.event_id)
print(CurrentTrigger.actor)
print(CurrentTrigger.action)
print(CurrentTrigger.rationale)


Iran
Maintain the declared closure of the Strait of Hormuz while intensifying calibrated asymmetric attacks on coalition naval, logistical, and regional support assets, using Iranian forces and aligned armed groups to impose costs without initiating unrestricted attacks on civilian shipping.
Iran could seek to demonstrate that US strikes and expanded escorts cannot restore normal maritime access at acceptable cost. Sustained but calibrated pressure would preserve coercive leverage, reinforce deterrence, and improve Iran’s bargaining position while retaining some control over the level of escalation.


In [ ]:
def resetWorld():
    global ActiveChanakya, Candidates
    del W[1:]
    E.clear()
    ActiveChanakya = None
    Candidates = []

    print("World reset to W0.")
# ------------------------
resetWorld()

World reset to W0.


#ChronoTantra | ChronoYantra | ChronoMantra | ChronoSutra
 [More Information ](http://bit.ly/chronobooks) <br>

[![More Information](https://raw.githubusercontent.com/prithwis/parashar21/refs/heads/main/images/CTYMS-Colab-Footer.png)](https://chronos.yantrajaal.com)